# Full-Duplex-Bench Vietnamese Evaluation Pipeline

Notebook này hướng dẫn chi tiết cách chạy thử nghiệm đánh giá hệ thống đàm thoại song song (Full-Duplex Spoken Dialogue) bằng tiếng Việt sử dụng hạ tầng GPU trên Kaggle hoặc Google Colab.

## 1. Clone Source Code từ GitHub

**Lưu ý:** Dự án này sử dụng branch `LamKD` từ repository `https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git`.

In [ ]:
# Di chuyển về thư mục gốc /kaggle/working trước khi clone để tránh bị đệ quy khi chạy lại cell nhiều lần
import os
import shutil
if os.path.exists('/kaggle/working'):
    %cd /kaggle/working
elif os.path.exists('/content'):
    %cd /content

# Xóa thư mục cũ nếu có để tránh clone lồng nhau
if os.path.exists('Full-Duplex-Bench'):
    shutil.rmtree('Full-Duplex-Bench')

!git clone -b LamKD https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git
%cd Full-Duplex-Bench/v1_v1.5

## 2. Cài đặt các thư viện hệ thống và thư viện Python

Bật GPU T4 trên Kaggle trước khi chạy cell này.

In [ ]:
# Cài đặt ffmpeg cho xử lý audio
!apt-get update && apt-get install -y ffmpeg

# Cài đặt các thư viện của Benchmark gốc và module sinh dữ liệu tự động
!pip install -r requirements.txt
!pip install -r data_generation/requirements.txt

## 3. Sinh dữ liệu Tiếng Việt Tự động (Dataset Generation)

Chạy các script tạo dữ liệu cho cả v1.0 và v1.5 từ các kịch bản JSON mẫu tiếng Việt đã chuẩn hóa.

In [ ]:
# Sinh dữ liệu v1.0 (Turn-Taking)
!python data_generation/v1_0/generate_v1_0.py

# Sinh dữ liệu v1.5 (Overlap)
!python data_generation/v1_5/generate_v1_5.py

## 4. Thiết lập khóa API Keys

Thay thế các giá trị bên dưới bằng API Key thực tế của bạn để tương tác với Gemini và GPT-4o phục vụ đánh giá hành vi.

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

## 5. Tương tác thử nghiệm với Agent (Model Inference)

Stream các tệp dữ liệu Tiếng Việt vừa tạo tới Agent và ghi âm lại các câu trả lời (`output.wav`). Ở đây ví dụ thử nghiệm với mô hình Gemini 2.5 Native Audio trên task `user_interruption` (v1.5).

In [ ]:
# Chạy sinh phản hồi của Agent khi nhận luồng âm thanh ngắt lời
!python model_inference/gemini/inference_gemini25_native.py \
    --base-dir dataset/v1_5 \
    --task user_interruption \
    --overwrite

## 6. Thực hiện bóc băng ASR (Yêu cầu GPU CUDA)

Chạy mô hình NeMo ASR lấy chi tiết word-level timestamps của Agent phục vụ các bước đo đạc độ trễ và phân tích hành vi.

In [ ]:
# Bóc băng câu trả lời của Agent khi có sự kiện nói chen
!python get_transcript/asr.py \
    --root_dir dataset/v1_5/user_interruption \
    --audio_name output.wav

# Bóc băng câu trả lời của Agent ở môi trường sạch đối chứng (cho v1.5)
!python get_transcript/asr.py \
    --root_dir dataset/v1_5/user_interruption \
    --audio_name clean_output.wav

## 7. Thực hiện đánh giá kết quả (Evaluation)

Chạy các kịch bản đánh giá để tổng hợp các chỉ số.

In [ ]:
%cd evaluation
print("=== 1. Đánh giá phân loại hành vi ứng xử (Behavior Categories) ===")
!python evaluate.py --task behavior --root_dir ../dataset/v1_5/user_interruption

print("\n=== 2. Đánh giá chất lượng và sự biến đổi giọng nói (Acoustic Features) ===")
!python evaluate.py --task general_before_after --root_dir ../dataset/v1_5/user_interruption
%cd ..